# ClearCLIP position-bias project — Colab runner

Runs the pipeline in `research-project/` without needing a local GPU.

**Before running:** on your machine, zip the repo's code (not the data/cache
folders, they don't exist yet) and upload it here:

```
cd research-project
zip -r code.zip src run.py config.yaml pyproject.toml
```

Then in Colab: **Runtime → Change runtime type → GPU** (T4 is fine), run the
cells below top to bottom. `extract` needs the GPU; `diagnose` /
`calibrate` / `evaluate` are CPU-only and will still work if the GPU runtime
gets reclaimed mid-session — just switch back to a CPU runtime and re-run
from the 'CPU-only stages' cell.

In [ ]:
!pip install -q open_clip_torch pyyaml scipy tqdm
import torch
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# Upload code.zip (see markdown cell above for how to create it)
from google.colab import files
uploaded = files.upload()  # pick code.zip
!mkdir -p /content/research-project
!unzip -oq code.zip -d /content/research-project
%cd /content/research-project

In [ ]:
# Download PASCAL VOC2012 (segmentation benchmark, ~2GB).
# The official host is sometimes slow; if this cell stalls for more than a
# few minutes, cancel it and instead upload the VOCtrainval tar from a
# Kaggle/Drive mirror you already have, then skip straight to the tar -xf line.
!mkdir -p data/raw
!wget -q --show-progress -P data/raw http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar
!tar -xf data/raw/VOCtrainval_11-May-2012.tar -C data/raw
!ls data/raw/VOCdevkit/VOC2012

## GPU stage — feature extraction (Phase 0)

One forward pass over the VOC20 val split (1449 images), cached to
`data/cache/voc20/*.npz`. Takes a few minutes on a T4.

In [ ]:
!python run.py --config config.yaml --stage extract

## CPU-only stages (Phases 1–3) — safe to re-run any time, no GPU required

`diagnose` is the go/no-go checkpoint: check `results/voc20_diagnostic.json`'s
`go_no_go` field before spending time on `calibrate`/`evaluate`.

In [ ]:
!python run.py --config config.yaml --stage diagnose

In [ ]:
!python run.py --config config.yaml --stage calibrate
!python run.py --config config.yaml --stage evaluate

In [ ]:
# Persist the cache + results so a killed/expired Colab session doesn't lose
# them — mount Drive once and copy out.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/clearclip_position_bias
!cp -r data/cache results /content/drive/MyDrive/clearclip_position_bias/
print('saved to /content/drive/MyDrive/clearclip_position_bias/')